# [Optional/exploratory] 1000G-projection cross-check

Sanity-checks `01_ancestry_pca_filter.ipynb`'s classification (AoU's own premade PCs) against an independent method: project the same AoU samples onto a genuine from-scratch 1000G PCA and see if the two methods agree on who lands where.

Reads `explore_1kg_reference.ipynb`'s 1000G PCA and `explore_hm3_ancestry_panel.ipynb`'s merged HM3 ancestry panel -- both already built. Not part of the main pipeline; run this only if you want the cross-check, nothing downstream depends on its output.

## Inputs

In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
CDR_VERSION = "v9"
PROJECT_DIR = "phenotypic_covariance_v9"

ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
FINAL_PCA_DIR = f"{ANCESTRY_BUCKET_DIR}/ancestry_pca_filter/final_pca"   # 01_ancestry_pca_filter.ipynb's keep-lists

KG_DIR = f"{WORKSPACE_BUCKET}/1000g_reference"   # explore_1kg_reference.ipynb's output
KG_PCA_PREFIX = f"{KG_DIR}/1kg_all_pca"           # whole-cohort fit -- broadest cross-check

HM3_PANEL_DIR = f"{ANCESTRY_BUCKET_DIR}/ancestry_panel"   # explore_hm3_ancestry_panel.ipynb's output
HM3_PANEL_PREFIX = f"{HM3_PANEL_DIR}/ancestry_panel_hm3_{CDR_VERSION}"

for ext in ("pgen", "pvar", "psam"):
    assert os.path.isfile(f"{HM3_PANEL_PREFIX}.{ext}"), (
        f"missing {HM3_PANEL_PREFIX}.{ext} -- run explore_hm3_ancestry_panel.ipynb first"
    )
for ext in ("eigenvec.allele", "acount"):
    assert os.path.isfile(f"{KG_PCA_PREFIX}.{ext}"), (
        f"missing {KG_PCA_PREFIX}.{ext} -- run explore_1kg_reference.ipynb first"
    )

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_1kg_crosscheck")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
OUT_PREFIX = os.path.join(LOCAL_WORK_DIR, f"aou_projected_1kg_{CDR_VERSION}")

print(HM3_PANEL_PREFIX)
print(KG_PCA_PREFIX)

## Project AoU samples onto the 1000G PCA

Same harmonize+`--score` pattern the retired `03_round2_1000g_filter.ipynb` used -- ID+REF+ALT `comm -12` harmonization, then plink2 `--score` against the 1000G PCA's own allele weights.

In [ ]:
def run_bash(script):
    subprocess.run(["bash", "-c", "set -e\n" + script], check=True)

run_bash(f'''
    plink2 --pfile "{HM3_PANEL_PREFIX}" --max-alleles 2 --rm-dup exclude-all --make-pgen --out "{OUT_PREFIX}_biallelic"

    grep -v '^##' "{OUT_PREFIX}_biallelic.pvar" | awk 'NR>1 {{print $3, $4, $5}}' | LC_ALL=C sort > "{OUT_PREFIX}_hm3_id_ref_alt.sorted"
    awk 'NR>1 {{print $2, $3, $4}}' "{KG_PCA_PREFIX}.acount" | LC_ALL=C sort > "{OUT_PREFIX}_kg_id_ref_alt.sorted"
    LC_ALL=C comm -12 "{OUT_PREFIX}_hm3_id_ref_alt.sorted" "{OUT_PREFIX}_kg_id_ref_alt.sorted" | awk '{{print $1}}' > "{OUT_PREFIX}_agreeing_snps.ids"
    echo "Agreeing with 1000G (ID+REF+ALT): $(wc -l < "{OUT_PREFIX}_agreeing_snps.ids")"

    WEIGHTS="{KG_PCA_PREFIX}.eigenvec.allele"
    HEADER=$(head -1 "$WEIGHTS")
    ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
    A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
    PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
    PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

    plink2 --pfile "{OUT_PREFIX}_biallelic" --extract "{OUT_PREFIX}_agreeing_snps.ids" --nonfounders \
      --read-freq "{KG_PCA_PREFIX}.acount" \
      --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize list-variants \
      --score-col-nums "${{PC1_COL}}-${{PC_LAST_COL}}" \
      --out "{OUT_PREFIX}"
''')

## Compare against `01_ancestry_pca_filter.ipynb`'s classification

Color each AoU sample's 1000G-projected PC1/PC2 by which premade-PC `SAMPLE_SET` it belongs to -- the two methods agree to the extent the colors form clean, separated clusters rather than a random scatter.

In [ ]:
projected = pd.read_csv(f"{OUT_PREFIX}.sscore", sep=r"\s+")
id_col = "#IID" if "#IID" in projected.columns else "IID"
projected = projected.rename(columns={id_col: "person_id"})
projected["person_id"] = projected["person_id"].astype(str)

SAMPLE_SETS = {
    "eur_strict": "p5",
    "eur_base":   "p10",
    "eur_loose":  "p50",
    "uniform": "uniform",
    "afr":        "p1",
    "eas":        "p50",
}

fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(projected["PC1_AVG"], projected["PC2_AVG"], s=2, alpha=0.1, color="lightgray", label="all projected")

for sample_set, prob_tag in SAMPLE_SETS.items():
    keep_path = os.path.join(FINAL_PCA_DIR, sample_set, f"final_keep_ids_{sample_set}_{prob_tag}.txt")
    if not os.path.isfile(keep_path):
        print(f"skipping {sample_set}: {keep_path!r} not found")
        continue
    keep_ids = set(pd.read_csv(keep_path, header=None)[0].astype(str))
    sub = projected[projected["person_id"].isin(keep_ids)]
    ax.scatter(sub["PC1_AVG"], sub["PC2_AVG"], s=3, alpha=0.4, label=sample_set)

ax.set_xlabel("1000G-projected PC1")
ax.set_ylabel("1000G-projected PC2")
ax.set_title("Premade-PC SAMPLE_SETs, viewed in an independent 1000G projection")
ax.legend(fontsize=7, markerscale=2, loc="best")
plt.tight_layout()
plot_path = os.path.join(FINAL_PCA_DIR, "crosscheck_1kg_projection.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")